# Step 2 (Approach 3): Train Two-Tower Bi-Encoder & Run FAISS Candidate Blocking

This notebook fine-tunes the **Two-Tower SentenceTransformer Bi-Encoder** model on SageMaker GPU using MultipleNegativesRankingLoss, extracts 384-dimensional normalized dense vectors for all test entities, runs country-partitioned FAISS blocking, and exports `candidate_pairs.tsv`.

In [ ]:
# 1. Install required packages directly into active SageMaker kernel
%pip install -q torch transformers sentence-transformers faiss-cpu catboost boto3 tqdm unidecode indic-transliteration

import os
import sys
import pandas as pd
import torch

# Ensure student_resource, approach_3_biEncoder, and current working directories are on sys.path
current_dir = os.path.dirname(os.path.abspath("")) if os.path.abspath("") else os.getcwd()
student_resource_dir = os.path.dirname(current_dir) if ("approach_" in os.path.basename(current_dir) or "global_" in os.path.basename(current_dir)) else current_dir

for p in [student_resource_dir, current_dir]:
    if p and p not in sys.path:
        sys.path.insert(0, p)

try:
    from approach_3_biEncoder.config import path_config, model_config
    from approach_3_biEncoder.src.trainer import train_bi_encoder
    from approach_3_biEncoder.src.blocking import run_blocking_pipeline
    from approach_3_biEncoder.src.s3_utils import upload_file_to_s3, sync_directory_from_s3
except ImportError:
    from config import path_config, model_config
    from src.trainer import train_bi_encoder
    from src.blocking import run_blocking_pipeline
    from src.s3_utils import upload_file_to_s3, sync_directory_from_s3

# Ensure dataset exists locally; if not, download automatically from S3 bucket!
if not os.path.exists(path_config.dataset_dir) or not os.path.exists(path_config.train_dir):
    print(f"Local dataset not found at {path_config.dataset_dir}. Syncing dataset from S3: s3://{path_config.s3_bucket}/dataset/...")
    sync_directory_from_s3(bucket=path_config.s3_bucket, s3_prefix="dataset", local_dir=path_config.dataset_dir)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 1. Train Bi-Encoder Model on GPU
Fine-tunes the two-tower SentenceTransformer backbone using MultipleNegativesRankingLoss on positive ground-truth pairs.

In [ ]:
print(f"Fine-tuning Bi-Encoder for {model_config.epochs} epochs on device: {model_config.device}...")

model_path = train_bi_encoder(
    epochs=model_config.epochs,
    batch_size=model_config.batch_size,
    learning_rate=model_config.learning_rate,
    save_s3=True
)

print(f"--> Saved fine-tuned Bi-Encoder model to: {model_path}")

## 2. Encode Test Set & Run FAISS Candidate Blocking
Passes test entities through fine-tuned Bi-Encoder, queries country-partitioned FAISS vector indices, and writes `candidate_pairs.tsv`.

In [ ]:
print("Loading test datasets...")
test_s1 = pd.read_csv(os.path.join(path_config.test_dir, "test_source1.tsv"), sep="\t")
test_s2 = pd.read_csv(os.path.join(path_config.test_dir, "test_source2.tsv"), sep="\t")
test_s3 = pd.read_csv(os.path.join(path_config.test_dir, "test_source3.tsv"), sep="\t")

candidate_tsv_path, candidate_map = run_blocking_pipeline(
    model_path=model_path,
    s1_df=test_s1,
    s2_df=test_s2,
    s3_df=test_s3,
    output_candidate_path=os.path.join(path_config.output_dir, "candidate_pairs.tsv")
)

# Upload candidate_pairs.tsv to S3
s3_key = f"{path_config.s3_prefix}/output/candidate_pairs.tsv"
upload_file_to_s3(candidate_tsv_path, path_config.s3_bucket, s3_key)

print(f"\n--> STEP 2 COMPLETE: candidate_pairs.tsv exported successfully for {len(test_s1):,} S1 entities!")
print("Next step: Open global_notebooks/03_global_feature_extraction_and_reranking.ipynb to score and format final submission!")